# 📊 HR Analytics — AtliQ Technologies
**Attendance Analysis | Apr–Jun 2022**

**Tools Used:** Python, Pandas, Matplotlib, Seaborn  
**Dataset:** AtliQ Employee Attendance Sheet (Apr–Jun 2022)  
**Goal:** Analyze employee attendance patterns, WFH trends, and leave behaviour to derive actionable HR insights.

---

## 1. Import Libraries

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded ✅')

## 2. Load & Clean Data

In [ ]:
# If using Google Colab, uncomment below to upload:
# from google.colab import files
# uploaded = files.upload()
# FILE = list(uploaded.keys())[0]

FILE = 'Attendance-Sheet-2022-2023.xlsx'  # update path if needed
sheets = ['Apr 2022', 'May 2022', 'June 2022']
frames = []

for sheet in sheets:
    df = pd.read_excel(FILE, sheet_name=sheet, header=1)
    df.columns = [str(c).strip() for c in df.columns]
    df = df.rename(columns={df.columns[0]: 'Employee_Code', df.columns[1]: 'Name'})
    df = df[df['Employee_Code'].notna()]
    df = df[~df['Employee_Code'].str.contains('Employee', na=True)]
    keep = ['Employee_Code', 'Name', 'TPD', 'P', 'WFH', 'PL', 'SL', 'LWP', 'WO']
    keep = [c for c in keep if c in df.columns]
    df = df[keep].copy()
    df['Month'] = sheet
    frames.append(df)

data = pd.concat(frames, ignore_index=True)

for c in ['TPD', 'P', 'WFH', 'PL', 'SL', 'LWP', 'WO']:
    if c in data.columns:
        data[c] = pd.to_numeric(data[c], errors='coerce').fillna(0)

print(f'Total Records: {len(data)}')
print(f'Unique Employees: {data["Name"].nunique()}')
data.head()

## 3. Summary Statistics

In [ ]:
monthly = data.groupby('Month')[['P','WFH','TPD','PL','SL','LWP']].sum().reindex(sheets)
monthly['Office_%'] = (monthly['P'] / monthly['TPD'] * 100).round(1)
monthly['WFH_%']    = (monthly['WFH'] / monthly['TPD'] * 100).round(1)
print('Monthly Summary:')
monthly[['P','WFH','PL','SL','Office_%','WFH_%']]

## 4. Chart 1 — Office Attendance vs WFH % by Month

In [ ]:
C = dict(primary='#1B4F72', accent='#2E86C1', green='#1E8449',
         orange='#E67E22', red='#C0392B', bg='#F4F6F7')
plt.rcParams.update({'font.family':'DejaVu Sans','axes.facecolor':C['bg'],
                     'figure.facecolor':'white','axes.grid':True,'grid.alpha':0.3,
                     'axes.spines.top':False,'axes.spines.right':False})

fig, ax = plt.subplots(figsize=(9, 5))
x = range(3)
b1 = ax.bar(x, monthly['Office_%'], width=0.35, label='Office %', color=C['primary'], zorder=3)
b2 = ax.bar([i+0.37 for i in x], monthly['WFH_%'], width=0.35, label='WFH %', color=C['accent'], zorder=3)
for b in list(b1)+list(b2):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.8,
            f"{b.get_height():.1f}%", ha='center', fontsize=10, fontweight='bold')
ax.set_xticks([i+0.18 for i in x]); ax.set_xticklabels(sheets, fontsize=12)
ax.set_ylabel('Percentage (%)'); ax.set_ylim(0, 105)
ax.set_title('Office Attendance vs WFH % by Month', fontsize=14, fontweight='bold', pad=15)
ax.legend(fontsize=10)
plt.tight_layout(); plt.show()

## 5. Chart 2 — Leave Type Distribution

In [ ]:
leave = {'Paid Leave': data['PL'].sum(), 'Sick Leave': data['SL'].sum(),
         'WFH Days': data['WFH'].sum(), 'Leave w/o Pay': data['LWP'].sum()}
ls = pd.Series(leave).sort_values()

fig, ax = plt.subplots(figsize=(9, 5))
colors = [C['green'], C['red'], C['accent'], C['orange']]
bars = ax.barh(ls.index, ls.values, color=colors, zorder=3)
for b in bars:
    ax.text(b.get_width()+1, b.get_y()+b.get_height()/2,
            f"{int(b.get_width())} days", va='center', fontsize=10, fontweight='bold')
ax.set_xlabel('Total Days'); ax.set_xlim(0, ls.max()*1.2)
ax.set_title('Leave Type Distribution — Apr to Jun 2022', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout(); plt.show()

## 6. Chart 3 — Top 10 WFH Employees

In [ ]:
top_wfh = data.groupby('Name')['WFH'].sum().sort_values(ascending=False).head(10)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(top_wfh.index, top_wfh.values, color=C['accent'], zorder=3)
for b in bars:
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.1,
            str(int(b.get_height())), ha='center', fontsize=9, fontweight='bold')
ax.set_ylabel('WFH Days')
ax.set_title('Top 10 Employees by WFH Days (Apr–Jun 2022)', fontsize=14, fontweight='bold', pad=15)
ax.set_xticklabels(top_wfh.index, rotation=30, ha='right', fontsize=9)
plt.tight_layout(); plt.show()

## 7. Chart 4 — Sick Leave Trend by Month

In [ ]:
sl_m = data.groupby('Month')['SL'].sum().reindex(sheets)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(sheets, sl_m.values, marker='o', linewidth=2.5, color=C['red'], markersize=12, zorder=3)
ax.fill_between(sheets, sl_m.values, alpha=0.15, color=C['red'])
for i, (x, y) in enumerate(zip(sheets, sl_m.values)):
    ax.text(i, y+0.4, str(int(y)), ha='center', fontsize=12, fontweight='bold')
ax.set_ylabel('Sick Leave Days'); ax.set_ylim(0, sl_m.max()*1.3)
ax.set_title('Sick Leave Trend by Month', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout(); plt.show()

## 8. Chart 5 — Employee Attendance Heatmap

In [ ]:
pivot = data.pivot_table(index='Name', columns='Month', values='P', aggfunc='sum')
pivot = pivot.reindex(columns=sheets).fillna(0).head(20)

fig, ax = plt.subplots(figsize=(9, 8))
sns.heatmap(pivot, annot=True, fmt='.0f', cmap='YlGn', ax=ax,
            linewidths=0.5, linecolor='white', cbar_kws={'label': 'Days Present'})
ax.set_title('Employee Office Presence Heatmap (Days)', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Month'); ax.set_ylabel('Employee')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=8)
plt.tight_layout(); plt.show()

## 9. Key Insights

Based on the analysis of AtliQ employee attendance data from April to June 2022:

1. **WFH is growing** — Work from home usage increased month-over-month, suggesting a shift toward hybrid work culture.
2. **Paid Leave is the most common leave type** — Employees primarily use PL over other leave types, indicating healthy leave utilisation.
3. **Sick leave spiked in June** — A notable rise in sick leave in June could indicate seasonal illness or employee burnout.
4. **High attendance consistency** — Most employees maintained strong office attendance (70%+) across all three months.
5. **A few employees prefer full WFH** — The top WFH employees consistently worked remotely, which may warrant a hybrid policy review.

---
*Analysis by: [Your Name] | Tools: Python, Pandas, Matplotlib, Seaborn*